# Task 4: retrain the final GAN for 200 epochs (Kaggle run)

Retrains only the FINAL model with the configuration Optuna already chose
(`configs/task4_cgan_best.yaml`, committed in the repository) for 200 epochs instead of 100.
No new search. Expected time: about 70 minutes on the Kaggle GPU.

Same setup as before: GPU on, Internet on, `WANDB_API_KEY` secret attached
(and `GITHUB_TOKEN` if the repository is private).
The notebook ends by zipping `task4_retrain_outputs.zip` (checkpoint, ONNX file, ONNX check):
download it from the Output panel.

In [ ]:
# 1. Secrets -> environment variables
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "genai-a1"

REPO = "aleenababar04/genai-assignment-1"
try:
    token = secrets.get_secret("GITHUB_TOKEN")   # only needed for a private repo
    REPO_URL = f"https://{token}@github.com/{REPO}.git"
except Exception:
    REPO_URL = f"https://github.com/{REPO}.git"


In [ ]:
# 2. Get the code and install the few packages Kaggle does not ship
%cd /kaggle/working
!rm -rf genai-assignment-1
!git clone -q $REPO_URL genai-assignment-1
%cd genai-assignment-1
!git log --oneline -1
!pip install -q optuna wandb onnx onnxruntime pytorch-msssim python-dotenv gdown

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings."
!grep -n "epochs" configs/task4_cgan.yaml
!cat configs/task4_cgan_best.yaml


In [ ]:
# 3. Put FS2K at data/FS2K (from an attached Kaggle dataset, or from Google Drive)
import glob, shutil, subprocess, zipfile

os.makedirs("data", exist_ok=True)
found = glob.glob("/kaggle/input/**/anno_train.json", recursive=True)
if found:
    source = os.path.dirname(found[0])
    print("using attached dataset:", source)
    shutil.copytree(source, "data/FS2K", dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if not zips:
        # File id from the official FS2K repository's Google Drive link.
        subprocess.run(["gdown", "1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp", "-O", "data/FS2K.zip"], check=True)
        zips = ["data/FS2K.zip"]
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("data/FS2K")
!ls data/FS2K


In [ ]:
# 4. Official split + stratified validation, 128x128 cache. It must print 899 / 159 / 1046 pairs
#    and the three cache folders must then contain train_, val_ and test_ files.
!python -m src.data.prepare_fs2k
!ls -la data/fs2k_128


In [ ]:
# 5. Final training, 200 epochs, with the settings Optuna chose (about 70 minutes).
#    Writes checkpoints/task4_cgan.pt and a new W&B run named final-task4_cgan.
!python -m src.training.train_cgan --mode final --epochs 200


In [ ]:
# 6. ONNX export of the generator, verified against PyTorch
!python -m src.export.export_onnx --task task4


In [ ]:
# 7. Collect the results. Download task4_retrain_outputs.zip from the notebook's Output panel.
!mkdir -p /kaggle/working/task4_retrain_outputs/checkpoints /kaggle/working/task4_retrain_outputs/models /kaggle/working/task4_retrain_outputs/results
!cp checkpoints/task4_cgan.pt /kaggle/working/task4_retrain_outputs/checkpoints/
!cp models/task4_generator.onnx /kaggle/working/task4_retrain_outputs/models/
!cp report/results/task4_generator_onnx_check.json /kaggle/working/task4_retrain_outputs/results/
%cd /kaggle/working
!zip -qr task4_retrain_outputs.zip task4_retrain_outputs
!ls -lh task4_retrain_outputs.zip


In [ ]:
# 8. OPTIONAL: test evaluation on Kaggle. It is not needed (the evaluation is repeated on the laptop),
#    and it failed here last time, so it comes after the zip.
%cd /kaggle/working/genai-assignment-1
!python -m src.evaluation.eval_cgan
